# 균열 방향 특징 융합 실험 (Colab)

**목적**: 교수님 자문 "역학 정보는 convolution 하지 말고 별도 feature로 넣어라"를 구현하고, 개선 여부를 수치로 검증한다.

**배경**
- 구조공학: 대각선(전단) 균열은 경고 없는 취성 파괴 신호, 수직(휨) 균열보다 위험
- 논문: 균열 방향(orientation)으로 전단/휨을 구분하고 손상을 지수화하는 방식이 보고됨
- 사전 검증(로컬): 탐지된 균열 영역의 **대각선 비율**이 우수 0.135 → 불량 0.218 로 61% 높음

**설계 (late fusion)**
```
이미지 ─┬─ 분류 CNN ────────────→ 등급 확률 3개 ─┐
        └─ 탐지 → 균열 영역 crop → 방향 특징 8개 ─┴─→ 로지스틱 회귀 → 최종 등급
```
CNN 단독(baseline)과 CNN+방향 특징을 **동일 검증셋**에서 비교한다.

**판정 기준**: 정확도가 아니라 **위험누락률**(실제 불량을 우수/보통으로 오판한 비율). 낮아져야 개선.

필요 파일 (My Drive 최상위): `house_grade_cls_v2.zip`, `grade_base_v2_best.pt`, 탐지모델 `best.pt`

## 1. 준비 — GPU, 데이터, 모델, 코드

In [ ]:
!nvidia-smi -L
%pip install -q ultralytics opencv-python-headless scikit-learn

from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile
ZIP = '/content/drive/MyDrive/house_grade_cls_v2.zip'
if not os.path.isdir('/content/house_grade_cls_v2'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')
DATA = '/content/house_grade_cls_v2'
for s in ('train', 'val'):
    print(s, {os.path.basename(d): len(os.listdir(d)) for d in sorted(glob.glob(f'{DATA}/{s}/*'))})

# 방향 특징 코드 (저장소에서 가져오기)
!git clone -q --depth 1 https://github.com/CalainKim/Google_AI_CapstoneDesign_Yolo-Based-Crack-Detection-and-Prediction-System.git /content/repo 2>/dev/null || true
import sys; sys.path.insert(0, '/content/repo/data-tools')
from orientation_features import orientation_features, FEATURE_KEYS
print('방향 특징:', FEATURE_KEYS)

In [ ]:
from ultralytics import YOLO
# 분류 모델(등급) + 탐지 모델(균열 위치)
CLS_PATH = '/content/drive/MyDrive/grade_base_v2_best.pt'
DET_PATH = '/content/drive/MyDrive/best.pt'   # 탐지 모델 (없으면 아래 셀에서 업로드)
assert os.path.exists(CLS_PATH), '분류 모델을 드라이브에 올려주세요'
assert os.path.exists(DET_PATH), '탐지 모델(best.pt)을 드라이브에 올려주세요'
cls_model = YOLO(CLS_PATH)
det_model = YOLO(DET_PATH)
print('클래스:', cls_model.names)

## 2. 특징 추출 — CNN 확률 + 방향 특징
탐지로 균열 영역을 찾아 그 안에서만 방향을 계산한다(배경 선 영향 제거). 균열 미탐지 시 방향 특징은 0으로 채우고 별도 플래그를 둔다.

In [ ]:
import numpy as np, cv2, tempfile, time

TMP = tempfile.mkdtemp()
KO = {'good': '우수', 'fair': '보통', 'poor': '불량'}
GROUPS = ['우수', '보통', '불량']

def crack_orientation(img_path):
    """탐지된 균열 영역들의 방향 특징 평균. 미탐지 시 None."""
    r = det_model.predict(img_path, imgsz=640, conf=0.15, verbose=False)[0]
    if not len(r.boxes):
        return None
    img = cv2.imread(img_path)
    if img is None:
        return None
    H, W = img.shape[:2]
    feats = []
    for b in r.boxes:
        x1, y1, x2, y2 = [int(v) for v in b.xyxy[0].tolist()]
        pad = 6
        crop = img[max(0, y1 - pad):min(H, y2 + pad), max(0, x1 - pad):min(W, x2 + pad)]
        if crop.size == 0 or min(crop.shape[:2]) < 20:
            continue
        p = f'{TMP}/c.jpg'
        cv2.imwrite(p, crop)
        d = orientation_features(p, mag_percentile=80)
        if d:
            feats.append([d[k] for k in FEATURE_KEYS])
    return np.mean(feats, axis=0) if feats else None

def build(split):
    """split 폴더 → (CNN확률 3, 방향특징 8, 미탐지플래그 1) 특징행렬과 정답"""
    X, y, meta = [], [], []
    files = [(KO[os.path.basename(d)], f)
             for d in sorted(glob.glob(f'{DATA}/{split}/*'))
             for f in sorted(glob.glob(f'{d}/*'))]
    t0 = time.time()
    for i, (label, f) in enumerate(files):
        r = cls_model.predict(f, imgsz=320, verbose=False)[0]
        probs = [float(r.probs.data[j]) for j in range(len(cls_model.names))]
        ori = crack_orientation(f)
        has = 1.0 if ori is not None else 0.0
        ori = ori if ori is not None else np.zeros(len(FEATURE_KEYS))
        X.append(probs + list(ori) + [has])
        y.append(label)
        if (i + 1) % 500 == 0:
            print(f'  {split} {i+1}/{len(files)}  ({time.time()-t0:.0f}s)')
    return np.array(X), np.array(y)

Xtr, ytr = build('train')
Xva, yva = build('val')
N_CLS = len(cls_model.names)
print('train', Xtr.shape, '| val', Xva.shape)
print('균열 탐지된 비율 — train %.1f%%, val %.1f%%' % (Xtr[:, -1].mean()*100, Xva[:, -1].mean()*100))
np.savez('/content/features.npz', Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva)  # 재실행 대비 저장

## 3. 평가 함수 — 위험누락률 중심

In [ ]:
from collections import defaultdict
results = {}

def report(y_true, y_pred, name):
    cm = defaultdict(int)
    for t, p in zip(y_true, y_pred):
        cm[(t, p)] += 1
    n = len(y_true)
    acc = sum(cm[(g, g)] for g in GROUPS) / n
    dtot = sum(cm[('불량', p)] for p in GROUPS)
    tp = cm[('불량', '불량')]
    miss = (dtot - tp) / dtot if dtot else 0
    fair_tot = sum(cm[('보통', p)] for p in GROUPS)
    print(f'\n===== {name} =====')
    print('실제\\예측 |', ' | '.join(f'{g:>5}' for g in GROUPS))
    for t in GROUPS:
        print(f'{t:>6}   |', ' | '.join(f'{cm[(t,p)]:>5}' for p in GROUPS))
    print(f'정확도 {acc:.1%} | 불량 재현율 {tp/dtot:.1%} | 위험누락률 {miss:.1%} | 보통 재현율 {cm[("보통","보통")]/fair_tot:.1%}')
    results[name] = (acc, tp/dtot, miss)
    return cm

## 4. 비교 — CNN 단독 vs CNN + 방향 특징

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

idx2ko = {i: KO[cls_model.names[i]] for i in range(N_CLS)}

# (A) CNN 단독 — 확률 최댓값 그대로 (기존 서비스와 동일)
pred_cnn = [idx2ko[int(np.argmax(x[:N_CLS]))] for x in Xva]
report(yva, pred_cnn, 'A. CNN 단독 (baseline)')

# (B) CNN 확률만 재보정 — 융합 효과와 분리하기 위한 대조군
clf_p = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
clf_p.fit(Xtr[:, :N_CLS], ytr)
report(yva, clf_p.predict(Xva[:, :N_CLS]), 'B. CNN 확률 재보정')

# (C) CNN + 방향 특징 융합 ← 우리의 개선
clf_f = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
clf_f.fit(Xtr, ytr)
report(yva, clf_f.predict(Xva), 'C. CNN + 방향 특징')

## 5. 최종 비교표 + 방향 특징의 기여도
발표 자료에 그대로 사용. 계수 부호로 어떤 방향이 위험 판정을 끌어올리는지 확인한다.

In [ ]:
print(f"{'모델':<24}{'정확도':>9}{'불량재현율':>11}{'위험누락률':>11}")
for k, (a, r, m) in results.items():
    print(f'{k:<24}{a:>8.1%}{r:>10.1%}{m:>10.1%}')

base_miss = results['A. CNN 단독 (baseline)'][2]
fuse_miss = results['C. CNN + 방향 특징'][2]
print(f'\n위험누락률 변화: {base_miss:.1%} → {fuse_miss:.1%} ({(fuse_miss-base_miss)*100:+.1f}%p)')
print('낮아졌으면 방향 특징이 안전 지표를 개선한 것.')

# 불량 판정에 기여하는 방향 특징 (로지스틱 회귀 계수)
names = [f'CNN_{cls_model.names[i]}' for i in range(N_CLS)] + FEATURE_KEYS + ['has_crack']
lr = clf_f.named_steps['logisticregression']
poor_i = list(lr.classes_).index('불량')
coef = lr.coef_[poor_i]
print('\n[불량 판정 기여도 상위]')
for i in np.argsort(-np.abs(coef))[:8]:
    print(f'  {names[i]:<16}{coef[i]:+.3f}')

## 6. 결과 저장
개선이 확인되면 융합 분류기를 저장해 서버에 반영한다.

In [ ]:
import joblib
joblib.dump({'model': clf_f, 'feature_names': names, 'classes': list(lr.classes_)},
            '/content/fusion_clf.joblib')
import shutil
shutil.copy('/content/fusion_clf.joblib', '/content/drive/MyDrive/fusion_clf.joblib')
from google.colab import files
files.download('/content/fusion_clf.joblib')
print('저장 완료')